# Hyperparameter search and special cases

HistGBM and LightGBM are indistinguishable at default settings, so both are tuned here and the main model is chosen from the tuned results.

## 1. Search design

Three parameters at three levels each, giving 27 configurations. The number of iterations is fixed at 300.

| HistGBM | LightGBM | Controls |
|---|---|---|
| `learning_rate` | `learning_rate` | step size |
| `max_leaf_nodes` | `num_leaves` | tree capacity |
| `min_samples_leaf` | `min_child_samples` | leaf size, regularisation |


**Only three fields are searched.**

Searching all eleven would take some 3,000 fits. Three representative fields are searched instead, the best configuration by mean rank is taken, and it is then verified across all eleven.
`creditors_total`,`fixed_assets`,`current_assets`

In [ ]:
from itertools import product
from pathlib import Path

import sys
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor

folder_01 = Path.cwd().parent / "01 EDA + Data PreProcessing"
sys.path.append(str(folder_01))

import data_prep as dp

pd.set_option("display.width", 220)

KEY = ["CompanyNumber_norm", "period_t", "period_t_plus_1"]
DATE_COLS = ["period_t", "period_t_plus_1", "available_date_t", "available_date_t_plus_1"]

N_SPLITS = 5
RANDOM_STATE = 0
N_PAIRS = 2_000_000

SEARCH_FIELDS = ["creditors_total", "fixed_assets", "current_assets"]

In [3]:
# Load
pairs = pd.read_csv("../01 EDA + Data PreProcessing/04_Five_CSV/03_financial_change_labels.csv",
                    dtype={"CompanyNumber_norm": str}, low_memory=False)
for c in DATE_COLS:
    pairs[c] = pd.to_datetime(pairs[c], errors="coerce")

meta = pd.read_csv("../01 EDA + Data PreProcessing/01_CompaniesSelected/UKcompanies_active_account_category_sample_100k.csv",
                   dtype={"CompanyNumber": str}, low_memory=False)
meta["CompanyNumber_norm"] = meta["CompanyNumber"].map(dp.normalise_company_number)
meta["IncorporationDate"] = pd.to_datetime(meta["IncorporationDate"], errors="coerce")

raw = pairs.merge(
    meta[["CompanyNumber_norm", "IncorporationDate", "CompanyCategory", "multi_sic_company"]],
    on="CompanyNumber_norm", how="left", validate="many_to_one")

df, _ = dp.clean_global(raw)

assignment = pd.read_csv("../01 EDA + Data PreProcessing/eda_output/split_assignment.csv",
                         dtype={"CompanyNumber_norm": str},
                         parse_dates=["period_t", "period_t_plus_1"])
df = df.merge(assignment, on=KEY, how="left", validate="one_to_one")
train = df[df["split_company"] == "train"].copy()

print(f"train: {len(train):,} rows")

train: 65,293 rows


In [ ]:
# Evaluation
def discrimination_rate(y_true, y_pred, n_pairs=N_PAIRS, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    i, j = rng.integers(0, n, n_pairs), rng.integers(0, n, n_pairs)
    answerable = (i != j) & (y_true[i] != y_true[j])
    if not answerable.any():
        return np.nan
    return float((y_pred[i[answerable]] != y_pred[j[answerable]]).mean())


def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if np.std(y_pred) < 1e-12:
        return {"discrim_rate": 0.0, "pairwise_acc": np.nan,
                "effective_acc": 0.5, "spearman": np.nan}
    
    tau = kendalltau(y_true, y_pred, variant="b").statistic
    acc = (tau + 1) / 2 if np.isfinite(tau) else np.nan
    disc = discrimination_rate(y_true, y_pred)
    return {
        "discrim_rate": disc, 
        "pairwise_acc": acc,
        "effective_acc": 0.5 + disc * (acc - 0.5) if np.isfinite(acc) else np.nan,
        "spearman": spearmanr(y_true, y_pred).statistic
    }

In [11]:
# Five-fold CV for one configuration

def cv_score(frame, target, model_name, params):
    """Effective accuracy for one configuration on one field, averaged over folds"""
    elig = frame[f"{target}_change_eligible"].fillna(False).astype(bool)
    sub = frame.loc[elig].copy()
    y = sub[f"{target}_signed_log_change"].astype(float)
    sub, y = sub.loc[y.notna()], y.loc[y.notna()]

    scores = []
    for tr, te in GroupKFold(N_SPLITS).split(sub, y, groups=sub["CompanyNumber_norm"]):
        X_tr, cat_cols, stats = dp.build_matrix(sub.iloc[tr], "gbm")
        X_te, _, _ = dp.build_matrix(sub.iloc[te], "gbm", fit_stats=stats)

        if model_name == "hgb":
            base = dict(loss="quantile", quantile=0.5, max_iter=300,
                        early_stopping=False, random_state=RANDOM_STATE,
                        categorical_features=[X_tr.columns.get_loc(c) for c in cat_cols])
            model = HistGradientBoostingRegressor(**{**base, **params})
        else:
            for c in cat_cols:
                X_tr[c] = X_tr[c].astype("category")
                X_te[c] = pd.Categorical(X_te[c], categories=X_tr[c].cat.categories)
            base = dict(objective="quantile", alpha=0.5, n_estimators=300,
                        verbose=-1, random_state=RANDOM_STATE)
            model = LGBMRegressor(**{**base, **params})

        model.fit(X_tr, y.iloc[tr])
        scores.append(evaluate(y.iloc[te], model.predict(X_te))["effective_acc"])
    return float(np.mean(scores))


def run_search(frame, model_name, grid, fields=SEARCH_FIELDS):
    """Grid search. Ranked within each field then averaged"""
    names = list(grid)
    rows = []
    for i, values in enumerate(product(*grid.values()), 1):
        params = dict(zip(names, values))
        row = {**params}
        for f in fields:
            row[f] = cv_score(frame, f, model_name, params)
        rows.append(row)
        print(f"  {i:2d}/{len(list(product(*grid.values())))}  {params}", flush=True)

    res = pd.DataFrame(rows)
    ranks = res[fields].rank(ascending=False)
    res["mean_rank"] = ranks.mean(axis=1)
    res["mean_acc"] = res[fields].mean(axis=1)
    return res.sort_values("mean_rank").reset_index(drop=True)


def report(res, default_params):
    """The top five, and where the default configuration ranks"""
    print("\ntop five:")
    print(res.head(5).round(4).to_string(index=False))

    mask = np.ones(len(res), bool)
    for k, v in default_params.items():
        mask &= (res[k] == v)
    dflt = res[mask]

    top5 = res.head(5)["mean_acc"]
    print(f"\nspread among top five: {top5.max() - top5.min():.4f}")
    if len(dflt):
        print(f"default rank: {int(dflt.index[0]) + 1}/{len(res)}  "
              f"mean_acc {dflt.iloc[0]['mean_acc']:.4f}  "
              f"vs best {res.iloc[0]['mean_acc']:.4f}")
    else:
        print("default is not in the grid")

## 2. HistGBM

default: `learning_rate=0.1`、`max_leaf_nodes=31`、`min_samples_leaf=20`

In [6]:
HGB_GRID = {
    "learning_rate":    [0.03, 0.1, 0.3],
    "max_leaf_nodes":   [15, 31, 63],
    "min_samples_leaf": [20, 50, 200],
}
HGB_DEFAULT = {"learning_rate": 0.1, "max_leaf_nodes": 31, "min_samples_leaf": 20}

hgb_search = run_search(train, "hgb", HGB_GRID)
report(hgb_search, HGB_DEFAULT)

   1/27  {'learning_rate': 0.03, 'max_leaf_nodes': 15, 'min_samples_leaf': 20}
   2/27  {'learning_rate': 0.03, 'max_leaf_nodes': 15, 'min_samples_leaf': 50}
   3/27  {'learning_rate': 0.03, 'max_leaf_nodes': 15, 'min_samples_leaf': 200}
   4/27  {'learning_rate': 0.03, 'max_leaf_nodes': 31, 'min_samples_leaf': 20}
   5/27  {'learning_rate': 0.03, 'max_leaf_nodes': 31, 'min_samples_leaf': 50}
   6/27  {'learning_rate': 0.03, 'max_leaf_nodes': 31, 'min_samples_leaf': 200}
   7/27  {'learning_rate': 0.03, 'max_leaf_nodes': 63, 'min_samples_leaf': 20}
   8/27  {'learning_rate': 0.03, 'max_leaf_nodes': 63, 'min_samples_leaf': 50}
   9/27  {'learning_rate': 0.03, 'max_leaf_nodes': 63, 'min_samples_leaf': 200}
  10/27  {'learning_rate': 0.1, 'max_leaf_nodes': 15, 'min_samples_leaf': 20}
  11/27  {'learning_rate': 0.1, 'max_leaf_nodes': 15, 'min_samples_leaf': 50}
  12/27  {'learning_rate': 0.1, 'max_leaf_nodes': 15, 'min_samples_leaf': 200}
  13/27  {'learning_rate': 0.1, 'max_leaf_nodes': 3

In [8]:
report(hgb_search, HGB_DEFAULT)


top five:
 learning_rate  max_leaf_nodes  min_samples_leaf  creditors_total  fixed_assets  current_assets  mean_rank  mean_acc
          0.03              31                50           0.5815        0.6015          0.5636     2.6667    0.5822
          0.03              63                20           0.5819        0.6033          0.5620     3.3333    0.5824
          0.03              31                20           0.5819        0.6004          0.5631     4.3333    0.5818
          0.03              63                50           0.5809        0.6037          0.5619     5.0000    0.5822
          0.03              31               200           0.5801        0.5981          0.5634     7.0000    0.5806

spread among top five: 0.0019
default rank: 6/27  mean_acc 0.5811  vs best 0.5822


## 3. LightGBM

default：`learning_rate=0.1`、`num_leaves=31`、`min_child_samples=20`

In [9]:
LGBM_GRID = {
    "learning_rate":     [0.03, 0.1, 0.3],
    "num_leaves":        [15, 31, 63],
    "min_child_samples": [20, 50, 200],
}
LGBM_DEFAULT = {"learning_rate": 0.1, "num_leaves": 31, "min_child_samples": 20}

lgbm_search = run_search(train, "lgbm", LGBM_GRID)

   1/27  {'learning_rate': 0.03, 'num_leaves': 15, 'min_child_samples': 20}
   2/27  {'learning_rate': 0.03, 'num_leaves': 15, 'min_child_samples': 50}
   3/27  {'learning_rate': 0.03, 'num_leaves': 15, 'min_child_samples': 200}
   4/27  {'learning_rate': 0.03, 'num_leaves': 31, 'min_child_samples': 20}
   5/27  {'learning_rate': 0.03, 'num_leaves': 31, 'min_child_samples': 50}
   6/27  {'learning_rate': 0.03, 'num_leaves': 31, 'min_child_samples': 200}
   7/27  {'learning_rate': 0.03, 'num_leaves': 63, 'min_child_samples': 20}
   8/27  {'learning_rate': 0.03, 'num_leaves': 63, 'min_child_samples': 50}
   9/27  {'learning_rate': 0.03, 'num_leaves': 63, 'min_child_samples': 200}
  10/27  {'learning_rate': 0.1, 'num_leaves': 15, 'min_child_samples': 20}
  11/27  {'learning_rate': 0.1, 'num_leaves': 15, 'min_child_samples': 50}
  12/27  {'learning_rate': 0.1, 'num_leaves': 15, 'min_child_samples': 200}
  13/27  {'learning_rate': 0.1, 'num_leaves': 31, 'min_child_samples': 20}
  14/27  {'l

In [10]:
report(lgbm_search, LGBM_DEFAULT)


top five:
 learning_rate  num_leaves  min_child_samples  creditors_total  fixed_assets  current_assets  mean_rank  mean_acc
          0.03          63                 20           0.5814        0.6040          0.5669     2.3333    0.5841
          0.03          31                 20           0.5819        0.6018          0.5668     2.6667    0.5835
          0.03          31                 50           0.5814        0.6013          0.5675     3.0000    0.5834
          0.03          63                 50           0.5807        0.6043          0.5667     3.3333    0.5839
          0.10          31                 50           0.5799        0.6014          0.5654     8.0000    0.5822

spread among top five: 0.0019
default rank: 7/27  mean_acc 0.5823  vs best 0.5841


## 4. Verifying the chosen configuration on all fields

**Why not take the top-ranked configuration.**

The top five differ by 0.0019 in both searches, less than the fold-to-fold standard deviation of
0.002–0.008 measured in `model_comparison.ipynb`. The ordering among them cannot be relied on.

**Only the learning rate shows a consistent direction.**

Every one of HistGBM's top five and LightGBM's top four uses a learning rate of 0.03, while the
default of 0.1 ranks 6th and 7th. The other two parameters alternate freely among the leading
configurations and show no preference.

A parameter that genuinely matters should see its best value recur among the leading configurations.
The learning rate does; the other two do not. The configuration therefore sets the learning rate to
0.03 and leaves the rest at their defaults.

**One explanation to rule out.**

With the number of iterations fixed at 300, cutting the learning rate to 0.03 reduces the total step
by about two thirds, so the gain may reflect insufficient capacity rather than the step size itself.
So we run 1,000 iterations on the three search fields to distinguish the two.

In [12]:
# Separating a smaller step from insufficient capacity
TUNED = {"learning_rate": 0.03}

for model_name, n_key in [("hgb", "max_iter"), ("lgbm", "n_estimators")]:
    print(f"\n{model_name}")
    for n_iter in (300, 1000):
        scores = {f: cv_score(train, f, model_name, {**TUNED, n_key: n_iter})
                  for f in SEARCH_FIELDS}
        print(f"  {n_key}={n_iter:4d}  " +
              "  ".join(f"{f}={v:.4f}" for f, v in scores.items()) +
              f"  mean={np.mean(list(scores.values())):.4f}")


hgb
  max_iter= 300  creditors_total=0.5819  fixed_assets=0.6004  current_assets=0.5631  mean=0.5818
  max_iter=1000  creditors_total=0.5826  fixed_assets=0.6003  current_assets=0.5635  mean=0.5821

lgbm
  n_estimators= 300  creditors_total=0.5819  fixed_assets=0.6018  current_assets=0.5668  mean=0.5835
  n_estimators=1000  creditors_total=0.5826  fixed_assets=0.6018  current_assets=0.5666  mean=0.5837


### Verification across all eleven fields

The chosen configuration is rerun on every field and compared against the default results from `model_comparison.ipynb`.

In [13]:
# The chosen configuration across all fields
FINAL_PARAMS = {"learning_rate": 0.03}

tuned = []
for model_name in ("hgb", "lgbm"):
    for target in dp.METRICS:
        tuned.append({"model": model_name, "target": target,
                      "tuned_acc": cv_score(train, target, model_name, FINAL_PARAMS)})
    print(f"done: {model_name}")

tuned = pd.DataFrame(tuned)

done: hgb
done: lgbm


In [14]:
# Against the default results
hgb = pd.read_csv("model_output/hgb_summary.csv")
lgbm = pd.read_csv("model_output/lgbm_summary.csv")

default = pd.concat([
    s[(s["model"] == name) & (~s["peer_pct"])][["target", "effective_acc", "n", "zero_frac"]]
     .assign(model=name)
    for name, s in [("hgb", hgb), ("lgbm", lgbm)]
], ignore_index=True).rename(columns={"effective_acc": "default_acc"})

comp = tuned.merge(default, on=["model", "target"], validate="one_to_one")
comp["gain"] = (comp["tuned_acc"] - comp["default_acc"]).round(4)

for model_name in ("hgb", "lgbm"):
    m = comp[comp["model"] == model_name].set_index("target")
    print(f"\n{model_name} median gain: {m['gain'].median():+.4f}  "
          f"fields improved: {int((m['gain'] > 0).sum())}/{len(m)}")
    print(m[["n", "zero_frac", "default_acc", "tuned_acc", "gain"]]
          .sort_values("gain", ascending=False).round(4).to_string())


hgb median gain: +0.0005  fields improved: 7/11
                                           n  zero_frac  default_acc  tuned_acc    gain
target                                                                                 
cash                                   25551     0.0413       0.5798     0.5855  0.0057
debtors                                21420     0.1122       0.5540     0.5580  0.0040
profit_loss                             2351     0.0332       0.5979     0.6009  0.0030
current_assets                         55886     0.0496       0.5609     0.5631  0.0022
net_current_assets_liabilities         59887     0.0549       0.5639     0.5656  0.0017
creditors_total                        59239     0.0656       0.5814     0.5819  0.0005
total_assets_less_current_liabilities  56348     0.0620       0.5698     0.5701  0.0003
net_assets_liabilities                 53706     0.0611       0.5595     0.5593 -0.0002
equity                                 63033     0.1282       0.5552   

## 5. Selecting the main model

Models are ranked within each field and the ranks averaged.

`profit_loss` is excluded

In [15]:
ranked = comp[comp["target"] != "profit_loss"].copy()
ranked["rank"] = ranked.groupby("target")["tuned_acc"].rank(ascending=False)

summary = ranked.groupby("model").agg(
    mean_rank=("rank", "mean"),
    rank_sd=("rank", "std"),
    n_best=("rank", lambda r: int((r == 1).sum())),
    mean_acc=("tuned_acc", "mean"),
).sort_values("mean_rank").round(4)

print(summary.to_string())

# per-field gap
pivot = ranked.pivot(index="target", columns="model", values="tuned_acc")
pivot["gap"] = (pivot["hgb"] - pivot["lgbm"]).round(4)
print("\nper-field gap (hgb-lgbm):")
print(pivot.sort_values("gap", key=abs, ascending=False).round(4).to_string())
print(f"\nlargest absolute gap: {pivot['gap'].abs().max():.4f}") # fold-to-fold SD of 0.002–0.008

       mean_rank  rank_sd  n_best  mean_acc
model                                      
lgbm         1.4   0.5164       6    0.5658
hgb          1.6   0.5164       4    0.5662

per-field gap (hgb-lgbm):
model                                     hgb    lgbm     gap
target                                                       
employees                              0.5229  0.5050  0.0179
debtors                                0.5580  0.5662 -0.0082
current_assets                         0.5631  0.5668 -0.0037
equity                                 0.5549  0.5566 -0.0017
fixed_assets                           0.6004  0.6018 -0.0014
total_assets_less_current_liabilities  0.5701  0.5712 -0.0010
net_assets_liabilities                 0.5593  0.5584  0.0009
net_current_assets_liabilities         0.5656  0.5650  0.0006
creditors_total                        0.5819  0.5819 -0.0000
cash                                   0.5855  0.5855  0.0000

largest absolute gap: 0.0179


#### Conclusions

**Insufficient capacity is ruled out**

Raising the iteration count to 1,000 improved the three-field mean by 0.0003 and 0.0002 respectively, with `fixed_assets` unchanged. Training has converged at 300 iterations, so the learning-rate effect is not a matter of total capacity. That performance did not fall also indicates overfitting is not the binding constraint.

**Across all fields: consistent in direction, but small**
LightGBM improves on 10 of 11 fields with a median of +0.0010; HistGBM on 7 of 11 with +0.0005. The
gains concentrate in the low-coverage fields and are near zero or negative on the well-covered ones.

---

**LightGBM is the main model**

After tuning, mean ranks are 1.4 and 1.6 with identical standard deviations, and LightGBM leads on 6
fields to 4. Mean accuracy points the other way (0.5662 against 0.5658).

The largest per-field gap is 0.0179, entirely from `employees`; excluding it, the largest is 0.0082 on `debtors` and the remaining nine fall within 0.0037, most around 0.001. Apart from `employees`, the two cannot be told apart.

Grounds for the choice:

1. Ten of eleven fields improved against seven, and six field wins against four. 
2. The `employees` gap.

---

**Final configuration**

```python
LGBMRegressor(objective="quantile", alpha=0.5,
              learning_rate=0.03, n_estimators=300,
              num_leaves=31, min_child_samples=20, # defaults
              verbose=-1, random_state=0)
```

The feature set is the base one, without relative-position features.

## 6. A two-part model for `employees`

**Why this experiment is needed**

`employees` is the only field where the tree model is significantly inferior to the linear model.

The gap comes entirely from the discrimination rate, not from accuracy when the models do commit:
the two GBMs return identical predictions for a large share of companies and thereby abstain.

The cause is zero-inflation: 73.3% of companies show no change in headcount. Quantile loss predicts
the leaf median, which is zero wherever most of the leaf has no change, so many leaves emit the same
value. Ridge's output is a continuous linear combination and cannot degenerate this way.

---

**Design**

hurdle model

| Sub-model | Training rows | Target | Output |
|---|---|---|---|
| A classification | all eligible samples | `y ≠ 0` or not | `p_change` |
| B regression | only samples show change | `signed_log_change` | `pred_size` |

`expected_change = p_change × pred_size`

The composite is the probability of change times its magnitude — the expected change. The classifier
outputs a continuous probability over every company, closing the abstention gap.


The regression sub-model is trained only on rows that changed but predicts on all test rows. Both
sub-models use the same fold assignment, or the composite would leak.

**Criterion**

The composite must be better than 0.5927 (Ridge), or Ridge is the better option for this field.

In [ ]:
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

TARGET = "employees"
BASELINE = 0.5927        # the Ridge figure from model_comparison.ipynb


def run_two_part(frame, target=TARGET, params=FINAL_PARAMS):
    """Five-fold CV for the two-part model
    Returns row-level p_change, pred_size and the composite"""
    elig = frame[f"{target}_change_eligible"].fillna(False).astype(bool)
    sub = frame.loc[elig].copy()
    y = sub[f"{target}_signed_log_change"].astype(float)
    sub, y = sub.loc[y.notna()], y.loc[y.notna()]

    moved = (y != 0).to_numpy()
    p_change = np.full(len(sub), np.nan)
    pred_size = np.full(len(sub), np.nan)
    pred_single = np.full(len(sub), np.nan)      # single-model reference

    for tr, te in GroupKFold(N_SPLITS).split(sub, y, groups=sub["CompanyNumber_norm"]):
        X_tr, cat_cols, stats = dp.build_matrix(sub.iloc[tr], "gbm")
        X_te, _, _ = dp.build_matrix(sub.iloc[te], "gbm", fit_stats=stats)
        for c in cat_cols:
            X_tr[c] = X_tr[c].astype("category")
            X_te[c] = pd.Categorical(X_te[c], categories=X_tr[c].cat.categories)

        # A classifier on all training rows
        clf = LGBMClassifier(n_estimators=300, verbose=-1,
                             random_state=RANDOM_STATE, **params)
        clf.fit(X_tr, moved[tr])
        p_change[te] = clf.predict_proba(X_te)[:, 1]

        # B regressor trained on changed rows only, predicting on every test row
        tr_moved = tr[moved[tr]]
        reg = LGBMRegressor(objective="quantile", alpha=0.5, n_estimators=300,
                            verbose=-1, random_state=RANDOM_STATE, **params)
        reg.fit(X_tr.loc[sub.index[tr_moved]], y.iloc[tr_moved])
        pred_size[te] = reg.predict(X_te)

        # single regressor, the tuned configuration
        single = LGBMRegressor(objective="quantile", alpha=0.5, n_estimators=300,
                               verbose=-1, random_state=RANDOM_STATE, **params)
        single.fit(X_tr, y.iloc[tr])
        pred_single[te] = single.predict(X_te)

    return pd.DataFrame({
        "y_true": y.to_numpy(), 
        "moved": moved,
        "p_change": p_change, 
        "pred_size": pred_size,
        "expected_change": p_change * pred_size,
        "pred_single": pred_single,
    }, index=sub.index)

In [17]:
tp = run_two_part(train)

In [ ]:
# Evaluate each layer separately
y_true = tp["y_true"].to_numpy()
m = tp["moved"].to_numpy()

print(f"classifier AUC: {roc_auc_score(tp['moved'], tp['p_change']):.4f}")
print(f"positive rate: {m.mean():.4f}\n")

rows = []
for name, pred in [
    ("regression on changed rows only", tp.loc[m, "pred_size"]),
    ("single model (tuned)", tp["pred_single"]),
    ("two-part composite", tp["expected_change"]),
]:
    truth = y_true[m] if "changed rows only" in name else y_true
    rows.append({"model": name, "n": len(truth), **evaluate(truth, pred)})

res = pd.DataFrame(rows).round(4)
print(res.to_string(index=False))
print(f"\nbaseline (Ridge): {BASELINE}")

classifier AUC: 0.7676
positive rate: 0.2669

                          model     n  discrim_rate  pairwise_acc  effective_acc  spearman
regression on changed rows only 16300        0.9954        0.7083         0.7073    0.5643
           single model (tuned) 61076        0.2019        0.5246         0.5050    0.0532
             two-part composite 61076        1.0000        0.6139         0.6139    0.2922

baseline (Ridge): 0.5927


#### Conclusions

**The two-part model resolves the abstention**

The composite exceeds the Ridge baseline by 0.0212 and the single regressor by 0.1089, both well
beyond the fold-to-fold standard deviation of 0.002–0.008. The discrimination rate rises to 1.0000:
the classifier emits a continuous probability for every company, so nothing is left unordered.

**The two sub-models**

The classifier reaches an AUC of 0.7676. The regressor, on the 16,300 rows that changed, reaches an
effective accuracy of 0.7073 and a Spearman of 0.5643 — the highest figures in the whole study. Once
the zero-change rows are removed, headcount change is the most predictable of all the fields; the single model's weakness was not a lack of signal but the collapse of leaf medians under 73.3% zero-inflation.

---

**Decision**

`employees` uses the two-part model; the other ten fields keep the single regressor (LightGBM，`learning_rate=0.03`).

```python
# classifier
LGBMClassifier(n_estimators=300, learning_rate=0.03, verbose=-1, random_state=0)
# regressor, trained on changed rows
LGBMRegressor(objective="quantile", alpha=0.5, n_estimators=300,
              learning_rate=0.03, verbose=-1, random_state=0)
# composite
expected_change = p_change * pred_size
```
